# LeetCode 2696 — Minimum String Length After Removing Substrings bằng Python

Link đề: https://leetcode.com/problems/minimum-string-length-after-removing-substrings/ · Độ khó: **Easy** · Chủ đề: String, Stack, Simulation

Skeleton LeetCode:

```python
class Solution:
    def minLength(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được **xoá** chuỗi con `"AB"` hoặc `"CD"` bao nhiêu lần tuỳ ý; sau mỗi lần xoá, hai phần còn lại **dính vào nhau** và có thể tạo ra `"AB"` / `"CD"` mới. Tìm độ dài **nhỏ nhất** có thể đạt.
2. Hiểu vì sao chỉ gọi `replace` **một lần** thì chưa đủ (các cặp "lồng" nhau như `AABB`).
3. Nắm cấu trúc **ngăn xếp (stack)** bằng `list` Python: `append`, `pop`, `stack[-1]`, kiểm tra rỗng.
4. Nhận dạng pattern kinh điển "**xoá cặp kề nhau sau khi dính lại**" ⇒ dùng stack.
5. Viết 4 cách: lặp `replace` đến khi hết xoá được, stack, con trỏ ghi tại chỗ (in-place), và stack tổng quát với bảng cặp bằng `dict`.
6. Chứng minh thuật toán đúng, phân tích độ phức tạp, đối chiếu nhiều input ngẫu nhiên có seed.
7. Tránh các lỗi quen thuộc: `replace` một lần, đếm `AB` / `CD` rồi trừ, quên kiểm tra stack rỗng.

## 2. Đề bài & ví dụ

Cho một chuỗi `s` gồm **chữ cái in hoa** tiếng Anh. Bạn có thể thực hiện thao tác sau **bất kỳ số lần nào**: xoá khỏi `s` **một chuỗi con** bằng `"AB"` hoặc bằng `"CD"`.

Lưu ý: sau khi xoá, **các phần còn lại được nối lại** thành một chuỗi mới, và có thể tạo ra những chuỗi con `"AB"` hoặc `"CD"` mới để xoá tiếp.

Trả về **độ dài nhỏ nhất** có thể đạt được của chuỗi sau khi xoá.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"ABFCACDB"` | `2` | xoá `AB` → `FCACDB`; xoá `CD` → `FCAB`; xoá `AB` → `FC`, độ dài 2 |
| 2 | `"ACBBD"` | `5` | không có `AB` hay `CD` làm chuỗi con nên không xoá được |

Constraints: `1 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh **in hoa**.

In [1]:
s = "ABFCACDB"
print("AB" in s, "CD" in s)          # True True: có thể xoá
s = s.replace("AB", "", 1)           # xoá AB đầu tiên
print(s)                             # FCACDB
s = s.replace("CD", "", 1)
print(s)                             # FCAB
s = s.replace("AB", "", 1)
print(s, len(s))                     # FC 2

print("ACBBD" .count("AB"), "ACBBD".count("CD"))   # 0 0: không xoá được -> độ dài 5

True True
FCACDB
FCAB
FC 2
0 0


## 3. Kiến thức nền

### 3.1. Chuỗi: `in`, `replace`, `len`

- `"AB" in s` kiểm tra `s` có chứa chuỗi con `"AB"` không.
- `s.replace(a, b)` thay **mọi** chỗ **không chồng nhau** của `a` bằng `b`, **tạo chuỗi mới** (chuỗi Python bất biến — không sửa tại chỗ). Thêm tham số thứ ba `count` để giới hạn số lần thay.
- Xoá chuỗi con = thay bằng chuỗi rỗng `""`.

In [2]:
s = "ABAB"
print("AB" in s)                 # True
print(s.replace("AB", ""))       # "" : cả hai AB bị xoá
print(repr("AABB".replace("AB", "")))   # 'AB' : chỉ xoá AB nằm giữa, AB mới hình thành chưa được xoá!
print(len("hello"), len(""))

True

'AB'
5 0


### 3.2. Stack (ngăn xếp) = "xếp chồng đĩa"

Stack là cấu trúc **vào sau, ra trước** (LIFO — Last In, First Out). Với `list` Python:

- `stack.append(x)`: **đặt** `x` lên đỉnh;
- `stack.pop()`: **lấy** phần tử ở đỉnh ra (và xoá khỏi stack);
- `stack[-1]`: **xem** phần tử đỉnh mà không lấy ra;
- `len(stack)` hoặc `if stack:` kiểm tra có rỗng không. **Chỉ được xem/lấy đỉnh khi stack không rỗng**, nếu không sẽ báo `IndexError`.

In [3]:
stack = []
stack.append("A")
stack.append("C")
print(stack, "đỉnh =", stack[-1])      # ['A', 'C'] đỉnh = C
print(stack.pop(), stack)              # C ['A']
print(bool(stack), len(stack))         # True 1
stack.pop()
print(bool(stack), len(stack))         # False 0 (rỗng)

try:
    stack[-1]
except IndexError as e:
    print("Lỗi khi xem đỉnh của stack rỗng:", e)

['A', 'C'] đỉnh = C
C ['A']
True 1
False 0
Lỗi khi xem đỉnh của stack rỗng: list index out of range


### 3.3. Duyệt từng ký tự và ghép cặp

`for ch in s:` duyệt từng ký tự. Với hai ký tự `a, b` ta có thể ghép thành chuỗi `a + b` rồi so với `"AB"`/`"CD"`, hoặc kiểm tra `(a, b)` có nằm trong một tập các cặp hợp lệ không.

In [4]:
for ch in "ABF":
    print(ch, end=" ")
print()
a, b = "A", "B"
print(a + b == "AB")                     # True
print((a, b) in {("A", "B"), ("C", "D")})   # True: kiểm tra cặp trong tập các cặp
print(a + b in ("AB", "CD"))             # True: kiểm tra chuỗi ghép trong tuple

A B F 
True
True
True


### 3.4. `dict` dùng làm bảng tra: "ký tự đóng → ký tự mở"

Nếu ta muốn tổng quát hoá (xoá cả `"XY"`, `"PQ"`...), có thể lưu bảng `{"B": "A", "D": "C"}`: khi gặp ký tự `B`, phần tử đứng trước nó mà ta mong đợi là `A`. `d.get(ch)` trả `None` nếu `ch` không có trong dict.

In [5]:
pair = {"B": "A", "D": "C"}
print(pair["B"], pair.get("D"), pair.get("Z"))   # A C None
print("B" in pair, "A" in pair)                   # True False

A C None
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đọc đề, ta tự nhiên nghĩ: "**cứ thấy `AB` hoặc `CD` thì xoá, xoá đến khi không còn**". Hãy làm tay với `s = "ABFCACDB"`:

1. Thấy `AB` ở đầu: xoá → `FCACDB`.
2. Thấy `CD` ở giữa: xoá → `FCAB`.
3. Thấy `AB` ở cuối: xoá → `FC`. Hết xoá được ⇒ độ dài `2`.

Điều đặc biệt: bước 2 **tạo ra** `AB` mới (từ `A` và `B` vốn bị `CD` ngăn cách). Nên sau mỗi lần xoá, **phải xét lại** chuỗi từ đầu — cái này đúng bản chất bài toán.

Một câu hỏi tự nhiên: **thứ tự xoá có ảnh hưởng đến độ dài cuối cùng không?** Ở mục 10 ta sẽ chứng minh là **không** (kết quả cuối cùng luôn duy nhất), nên chỉ cần xoá theo một thứ tự bất kỳ.

In [6]:
# thử xoá theo hai thứ tự khác nhau, kết quả cuối vẫn như nhau
s1 = "ABFCACDB".replace("CD", "", 1)       # xoá CD trước: ABFCAB
s1 = s1.replace("AB", "", 1)               # FCAB
s1 = s1.replace("AB", "", 1)               # FC
print("CD trước:", s1)

s2 = "ABFCACDB".replace("AB", "", 1)       # xoá AB trước: FCACDB
s2 = s2.replace("CD", "", 1)               # FCAB
s2 = s2.replace("AB", "", 1)               # FC
print("AB trước:", s2)

CD trước: FC
AB trước: FC


## 5. Cách 1 — Brute Force / baseline: lặp `replace` đến khi không đổi

**Ý tưởng:** lặp lại: nếu `s` còn chứa `"AB"` hoặc `"CD"` thì xoá **một** lần xuất hiện của nó rồi xét lại; dừng khi cả hai đều không còn. Trả về `len(s)`.

Cách này **trung thực đúng với đề** nên rất dễ tin tưởng; ta cũng dùng nó làm **oracle** để kiểm tra các cách nhanh hơn.

In [7]:
def min_length_replace(s: str) -> int:
    while "AB" in s or "CD" in s:
        s = s.replace("AB", "", 1)        # xoá AB đầu tiên (nếu có)
        s = s.replace("CD", "", 1)        # xoá CD đầu tiên (nếu có)
    return len(s)

print(min_length_replace("ABFCACDB"))   # 2
print(min_length_replace("ACBBD"))      # 5
print(min_length_replace("AABB"))       # 0  (lồng nhau)

2
5
0


**Độ phức tạp:** mỗi lần xoá xoá được 2 ký tự nên có tối đa `n/2` vòng; mỗi vòng `in` và `replace` đều quét chuỗi `O(n)`. Time `O(n²)`, Space `O(n)` (mỗi lần `replace` tạo chuỗi mới). Với `n ≤ 100` vẫn chạy ngay, nhưng đây là hướng **không tối ưu**.

**Nút thắt:** sau mỗi lần xoá ta **quét lại từ đầu** dù phần lớn chuỗi không đổi. Hãy xử lý **một lượt** từ trái sang phải và nhớ cái đã đọc.

## 6. Ý tưởng thuật toán / bước đột phá

Cặp cần xoá luôn là **hai ký tự kề nhau** `AB` hoặc `CD`. Hãy đọc chuỗi từ trái sang phải và **giữ lại các ký tự "chưa bị xoá"** trong một danh sách. Khi đọc ký tự mới `ch`:

- Nếu `ch == 'B'` và ký tự **cuối cùng đang giữ** là `'A'` ⇒ chúng tạo thành `AB` ⇒ **xoá cả hai** (bỏ `A` đang giữ, không giữ `B`).
- Tương tự `ch == 'D'` và ký tự cuối đang giữ là `'C'`.
- Còn lại ⇒ giữ `ch` lại.

"Ký tự cuối đang giữ" và "bỏ ký tự cuối" chính là thao tác **đỉnh stack** và `pop`. Nhờ vậy khi xoá một cặp, ký tự **trước đó** lộ ra làm "đỉnh" mới và tự động có thể ghép cặp với ký tự tiếp theo — đây là cách ta "dính lại sau khi xoá" mà **không phải quét lại** chuỗi.

Độ dài đáp án = số ký tự còn lại trong stack.

In [8]:
s = "ABFCACDB"
stack = []
for ch in s:
    if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
        removed = stack.pop()
        print(f"gặp {ch}, đỉnh {removed} -> xoá cặp {removed}{ch};  stack = {stack}")
    else:
        stack.append(ch)
        print(f"gặp {ch}, đẩy vào;              stack = {stack}")
print(len(stack))

gặp A, đẩy vào;              stack = ['A']
gặp B, đỉnh A -> xoá cặp AB;  stack = []
gặp F, đẩy vào;              stack = ['F']
gặp C, đẩy vào;              stack = ['F', 'C']
gặp A, đẩy vào;              stack = ['F', 'C', 'A']
gặp C, đẩy vào;              stack = ['F', 'C', 'A', 'C']
gặp D, đỉnh C -> xoá cặp CD;  stack = ['F', 'C', 'A']
gặp B, đỉnh A -> xoá cặp AB;  stack = ['F', 'C']
2


## 7. Các cách tiếp theo

Ta có Cách 1 (lặp `replace`) và sẽ xem thêm: (2) stack chuẩn, (3) con trỏ ghi tại chỗ (không cần `append/pop`), (4) stack tổng quát dùng `dict` (dễ mở rộng cho nhiều cặp).

### Cách 2 — Stack

Giải thích từng đoạn:

- `stack = []`: các ký tự còn lại sau khi đã xoá hết những cặp xoá được **trong phần đã đọc**.
- `for ch in s`: đọc từng ký tự theo thứ tự.
- `if stack and (...)`: **phải kiểm tra `stack` không rỗng trước** (`and` đoản mạch: nếu `stack` rỗng thì vế sau không được tính ⇒ không `IndexError`).
- `(stack[-1] == "A" and ch == "B")` hoặc `(stack[-1] == "C" and ch == "D")`: đỉnh + ký tự mới tạo thành `AB` hoặc `CD`.
- `stack.pop()`: xoá cặp (`A`/`C` lấy ra khỏi stack; `ch` không được đẩy vào).
- `else: stack.append(ch)`: không ghép cặp được ⇒ giữ lại.
- `return len(stack)`.

In [9]:
def min_length_stack(s: str) -> int:
    stack = []
    for ch in s:
        if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
            stack.pop()
        else:
            stack.append(ch)
    return len(stack)

print(min_length_stack("ABFCACDB"), min_length_stack("ACBBD"), min_length_stack("AABB"))

2 5 0


In [10]:
class SolutionStack:
    def minLength(self, s: str) -> int:
        stack = []
        for ch in s:
            if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
                stack.pop()
            else:
                stack.append(ch)
        return len(stack)

print(SolutionStack().minLength("ABFCACDB"))

2


**Độ phức tạp:** mỗi ký tự được đẩy vào tối đa một lần và lấy ra tối đa một lần ⇒ Time `O(n)`, Space `O(n)` (trường hợp xấu nhất không xoá được gì, stack chứa cả chuỗi).

### Cách 3 — Con trỏ ghi tại chỗ (in-place, "write pointer")

Stack có thể **mượn ngay mảng ký tự gốc** làm chỗ chứa: dùng chỉ số `top` là "kích thước stack hiện tại". Phần `arr[0:top]` chính là stack.

- Đọc ký tự `ch` tại `arr[i]`.
- Nếu `top > 0` và `arr[top - 1] + ch` là `AB` hoặc `CD` ⇒ `top -= 1` (pop).
- Ngược lại `arr[top] = ch; top += 1` (push).
- Cuối cùng đáp án là `top`.

Ưu điểm: **không cấp phát stack mới** — Space phụ `O(1)` nếu sửa trực tiếp trên mảng ký tự (ở đây `list(s)` là một bản sao nên tổng vẫn `O(n)`; trong C++/Java với mảng sửa được thì thật sự `O(1)`).

In [11]:
def min_length_inplace(s: str) -> int:
    arr = list(s)             # chuỗi Python bất biến, nên đổi sang list để ghi đè được
    top = 0                   # arr[:top] là "stack"
    for i in range(len(arr)):
        ch = arr[i]
        if top > 0 and arr[top - 1] + ch in ("AB", "CD"):
            top -= 1
        else:
            arr[top] = ch
            top += 1
    return top

print(min_length_inplace("ABFCACDB"), min_length_inplace("ACBBD"), min_length_inplace("AABB"))

2 5 0


In [12]:
class SolutionInPlace:
    def minLength(self, s: str) -> int:
        arr = list(s)
        top = 0
        for i in range(len(arr)):
            ch = arr[i]
            if top > 0 and arr[top - 1] + ch in ("AB", "CD"):
                top -= 1
            else:
                arr[top] = ch
                top += 1
        return top

print(SolutionInPlace().minLength("ABFCACDB"))

2


**Độ phức tạp:** một lượt `O(n)`; Space phụ `O(1)` nếu coi `arr` là dữ liệu vào được phép sửa (với `list(s)` ở Python là `O(n)`). Vì `top ≤ i` luôn đúng nên ghi `arr[top]` **không bao giờ ghi đè** phần chưa đọc `arr[i+1:]`.

### Cách 4 — Stack tổng quát bằng `dict` bảng cặp

Nếu đề đổi thành "xoá `XY`, `PQ`, `MN`..." thì ta không muốn viết 3 điều kiện `or`. Dùng `dict` ánh xạ **ký tự đóng → ký tự mở**: `closing = {"B": "A", "D": "C"}`. Điều kiện ghép cặp: `ch in closing and stack and stack[-1] == closing[ch]`. Muốn thêm cặp mới chỉ thêm một dòng vào dict.

In [13]:
def min_length_dict(s: str, closing=None) -> int:
    if closing is None:
        closing = {"B": "A", "D": "C"}
    stack = []
    for ch in s:
        if ch in closing and stack and stack[-1] == closing[ch]:
            stack.pop()
        else:
            stack.append(ch)
    return len(stack)

print(min_length_dict("ABFCACDB"), min_length_dict("ACBBD"))
print(min_length_dict("XYZPQ", {"Y": "X", "Q": "P"}))     # cặp khác: XY và PQ -> còn Z, độ dài 1

2 5
1


In [14]:
class SolutionDict:
    def minLength(self, s: str) -> int:
        closing = {"B": "A", "D": "C"}
        stack = []
        for ch in s:
            if ch in closing and stack and stack[-1] == closing[ch]:
                stack.pop()
            else:
                stack.append(ch)
        return len(stack)

print(SolutionDict().minLength("ABFCACDB"))

2


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (stack). Tra `dict` mỗi ký tự là `O(1)` trung bình.

## 8. TRACE từng bước

Trace Cách 2 (stack) với `s = "ABFCACDB"`:

| i | ký tự `ch` | đỉnh trước | có ghép cặp? | hành động | stack sau |
|---|---|---|---|---|---|
| 0 | A | (rỗng) | không | push A | `[A]` |
| 1 | B | A | `AB` ✓ | pop | `[]` |
| 2 | F | (rỗng) | không | push F | `[F]` |
| 3 | C | F | không | push C | `[F, C]` |
| 4 | A | C | không | push A | `[F, C, A]` |
| 5 | C | A | không | push C | `[F, C, A, C]` |
| 6 | D | C | `CD` ✓ | pop | `[F, C, A]` |
| 7 | B | A | `AB` ✓ | pop | `[F, C]` |

Cuối cùng stack có `2` phần tử ⇒ đáp án `2`. Chú ý ở bước 7: `B` ghép được với `A` **chỉ vì** `CD` đã được xoá ở bước 6 làm `A` lộ ra ở đỉnh.

In [15]:
def trace_stack(s):
    stack = []
    for i, ch in enumerate(s):
        top = stack[-1] if stack else "(rỗng)"
        if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
            action = f"ghép {top}{ch} -> pop"
            stack.pop()
        else:
            action = "push"
            stack.append(ch)
        print(f"i={i} ch={ch} đỉnh trước={top:<7} {action:<14} stack={stack}")
    print("độ dài =", len(stack))
    return len(stack)

trace_stack("ABFCACDB")

i=0 ch=A đỉnh trước=(rỗng)  push           stack=['A']
i=1 ch=B đỉnh trước=A       ghép AB -> pop stack=[]
i=2 ch=F đỉnh trước=(rỗng)  push           stack=['F']
i=3 ch=C đỉnh trước=F       push           stack=['F', 'C']
i=4 ch=A đỉnh trước=C       push           stack=['F', 'C', 'A']
i=5 ch=C đỉnh trước=A       push           stack=['F', 'C', 'A', 'C']
i=6 ch=D đỉnh trước=C       ghép CD -> pop stack=['F', 'C', 'A']
i=7 ch=B đỉnh trước=A       ghép AB -> pop stack=['F', 'C']
độ dài = 2


2

In [16]:
# Trace thứ hai: chuỗi lồng nhau hoàn toàn, xoá hết
trace_stack("AABB")
print()
# Trace thứ ba: ví dụ 2 của đề, không xoá được gì
trace_stack("ACBBD")

i=0 ch=A đỉnh trước=(rỗng)  push           stack=['A']
i=1 ch=A đỉnh trước=A       push           stack=['A', 'A']
i=2 ch=B đỉnh trước=A       ghép AB -> pop stack=['A']
i=3 ch=B đỉnh trước=A       ghép AB -> pop stack=[]
độ dài = 0

i=0 ch=A đỉnh trước=(rỗng)  push           stack=['A']
i=1 ch=C đỉnh trước=A       push           stack=['A', 'C']
i=2 ch=B đỉnh trước=C       push           stack=['A', 'C', 'B']
i=3 ch=B đỉnh trước=B       push           stack=['A', 'C', 'B', 'B']
i=4 ch=D đỉnh trước=B       push           stack=['A', 'C', 'B', 'B', 'D']
độ dài = 5


5

## 9. Code nộp LeetCode

Dùng Cách 2 (stack), đúng class `Solution` và method `minLength`.

In [17]:
class Solution:
    def minLength(self, s: str) -> int:
        stack = []
        for ch in s:
            if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
                stack.pop()
            else:
                stack.append(ch)
        return len(stack)

In [18]:
print(Solution().minLength("ABFCACDB"))    # 2
print(Solution().minLength("ACBBD"))       # 5

2
5


## 10. Vì sao thuật toán đúng

Cần chứng minh hai điều: (a) stack tạo ra **một chuỗi không còn `AB`/`CD`**, tức là xoá hết những gì xoá được; (b) độ dài đó là **nhỏ nhất**, tức không phụ thuộc thứ tự xoá.

**(a) Invariant của stack.** Sau khi đọc `s[0..i]`, stack chứa đúng chuỗi thu được khi ta xoá liên tục `AB`/`CD` khỏi `s[0..i]` cho tới khi không xoá được nữa, **và** stack không chứa `AB` hay `CD` làm chuỗi con. Chứng minh quy nạp: khi thêm ký tự `ch`, chuỗi con `AB`/`CD` mới (nếu có) chỉ có thể kết thúc ở `ch`, nên chỉ cần xét cặp (đỉnh stack, `ch`). Nếu ghép được ⇒ xoá, stack vẫn không có cặp (nó chỉ là stack cũ bỏ đi một phần tử). Nếu không ghép được ⇒ push, và vì cặp mới duy nhất có thể xuất hiện đã bị loại trừ nên vẫn không có cặp.

**(b) Kết quả duy nhất.** Xét hai lần xoá khác nhau có thể làm ngay trên cùng một chuỗi. Mỗi lần xoá chiếm **2 vị trí liền nhau**. Hai lần xoá không thể **chồng nhau** (dùng chung một vị trí), vì một ký tự chỉ có thể là *ký tự đầu* của cặp (`A`, `C`) hoặc *ký tự cuối* của cặp (`B`, `D`), không bao giờ là cả hai. Hai lần xoá ở các vị trí rời nhau thì **làm thứ tự nào cũng ra cùng một chuỗi** (chúng giao hoán). Quá trình xoá luôn dừng (mỗi lần xoá làm chuỗi ngắn đi 2). Một hệ thay thế luôn dừng mà hai bước bất kỳ đều "gộp lại được" thì mọi thứ tự xoá cho **cùng một chuỗi cuối cùng** (tính chất hội tụ). Vậy độ dài nhỏ nhất là duy nhất và bằng kích thước stack; ta cũng đã kiểm nghiệm điều này bằng thực nghiệm ở mục 12.4 (xoá cặp ngẫu nhiên vẫn cùng độ dài).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (tối đa 100).

| Cách | Time | Space |
|---|---|---|
| 1. lặp `replace` | `O(n²)` | `O(n)` |
| 2. stack | `O(n)` | `O(n)` |
| 3. con trỏ ghi tại chỗ | `O(n)` | `O(1)` phụ (mảng ký tự sửa được) |
| 4. stack + `dict` | `O(n)` | `O(n)` |

Cách 1: tối đa `n/2` vòng, mỗi vòng quét `O(n)`. Cách 2/3/4: mỗi ký tự được xử lý đúng một lần, `push`/`pop` đều `O(1)` (amortized).

## 12. Bộ test

### 12.1. Ví dụ của đề và các cách giải cần so sánh

In [19]:
sols = [SolutionStack(), SolutionInPlace(), SolutionDict(), Solution()]
funcs = [min_length_replace, min_length_stack, min_length_inplace, min_length_dict] + [x.minLength for x in sols]

for f in funcs:
    assert f("ABFCACDB") == 2
    assert f("ACBBD") == 5
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case

- Một ký tự: `"A"` → `1`.
- Đúng một cặp: `"AB"` → `0`, `"CD"` → `0`.
- Ngược thứ tự: `"BA"` → `2`, `"DC"` → `2` (không phải `AB`/`CD`).
- Lồng nhau: `"AABB"` → `0`, `"ACDB"` → `0`, `"CABD"` (xoá `AB` ở giữa, còn `CD` ⇒ `0`).
- Kề nhau: `"ABAB"` → `0`, `"ABCD"` → `0`, `"CDAB"` → `0`.
- Không ghép được cặp chéo: `"ACBD"` → `4` (A-C, C-B, B-D đều không phải cặp).
- Chuỗi dài 100: toàn `A` → `100`; `"AB" * 50` → `0`; `"A" * 50 + "B" * 50` → `0`.

In [20]:
cases = {
    "A": 1, "AB": 0, "CD": 0, "BA": 2, "DC": 2,
    "AABB": 0, "ACDB": 0, "CABD": 0, "ABAB": 0, "ABCD": 0, "CDAB": 0,
    "ACBD": 4, "ADBC": 4,
    "A" * 100: 100, "AB" * 50: 0, "A" * 50 + "B" * 50: 0,
    "Z": 1, "ABZ": 1, "ZAB": 1, "AZB": 3,
}
for f in funcs:
    for s, want in cases.items():
        assert f(s) == want, (f, s, want)
print("Edge case OK:", len(cases), "trường hợp x", len(funcs), "cách")

Edge case OK: 20 trường hợp x 8 cách


### 12.3. Phản ví dụ cho các cách làm sai

Hai cách làm hay gặp mà sai (mục 13): `replace` **một lượt** và **đếm** số `AB`/`CD`. Ta xác nhận chúng sai trên những input nhỏ.

In [21]:
def sai_replace_mot_lan(s):
    return len(s.replace("AB", "").replace("CD", ""))

def sai_dem(s):
    return len(s) - 2 * (s.count("AB") + s.count("CD"))

for s in ["AABB", "ACDB"]:
    print(s, "đúng =", Solution().minLength(s), "| replace 1 lần =", sai_replace_mot_lan(s), "| đếm =", sai_dem(s))
    assert sai_replace_mot_lan(s) != Solution().minLength(s)
    assert sai_dem(s) != Solution().minLength(s)

AABB đúng = 0 | replace 1 lần = 2 | đếm = 2
ACDB đúng = 0 | replace 1 lần = 2 | đếm = 2


### 12.4. Đối chiếu nhiều input ngẫu nhiên (có seed)

Sinh chuỗi hợp lệ theo constraints (độ dài `1..100`, toàn chữ in hoa). Để các cặp `AB`/`CD` xuất hiện nhiều, ta lấy ký tự từ bảng chữ nhỏ `"ABCD"` (dễ lồng nhau) và từ bảng `"ABCDEF"`; thêm cả chuỗi từ toàn bộ 26 chữ.

In [22]:
import random, string

random.seed(2696)
alphabets = ["AB", "ABCD", "ABCDEF", string.ascii_uppercase]
cnt = 0
for _ in range(1500):
    alpha = random.choice(alphabets)
    n = random.randint(1, 100)
    s = "".join(random.choice(alpha) for _ in range(n))
    want = min_length_replace(s)          # oracle: làm đúng theo đề
    for f in funcs:
        assert f(s) == want, (s, f, want)
    cnt += 1
print("OK", cnt, "chuỗi ngẫu nhiên; mọi cách khớp")

# kiểm tra "thứ tự xoá không quan trọng": xoá ngẫu nhiên một cặp mỗi lần, kết quả vẫn bằng stack
def random_order_length(s):
    while True:
        spots = [i for i in range(len(s) - 1) if s[i:i + 2] in ("AB", "CD")]
        if not spots:
            return len(s)
        i = random.choice(spots)
        s = s[:i] + s[i + 2:]

for _ in range(300):
    s = "".join(random.choice("ABCD") for _ in range(random.randint(1, 60)))
    assert random_order_length(s) == Solution().minLength(s)
print("Thứ tự xoá ngẫu nhiên luôn cho cùng độ dài")

OK 1500 chuỗi ngẫu nhiên; mọi cách khớp
Thứ tự xoá ngẫu nhiên luôn cho cùng độ dài


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ `replace` một lượt

Nghĩ rằng `s.replace("AB", "").replace("CD", "")` là xong. Nhưng xoá xong có thể tạo ra cặp **mới**.

In [23]:
def sai_replace_mot_luot(s: str) -> int:
    s = s.replace("AB", "")
    s = s.replace("CD", "")
    return len(s)

print("sai :", sai_replace_mot_luot("AABB"))      # 2
print("đúng:", Solution().minLength("AABB"))      # 0
assert sai_replace_mot_luot("ACDB") != Solution().minLength("ACDB")

sai : 2
đúng: 0


**Vì sao sai:** `"AABB".replace("AB", "")` chỉ xoá `AB` ở giữa, để lại `"AB"` — chính cặp mới này cần xoá tiếp. Và `"ACDB"`: sau `replace("AB")` không đổi (không có `AB`), `replace("CD")` đưa về `"AB"`, nhưng không ai xoá tiếp. Cách sửa: lặp đến khi hết đổi, hoặc dùng **stack**.

### Lỗi 2: Đếm số lần xuất hiện rồi trừ

In [24]:
def sai_dem_cap(s: str) -> int:
    return len(s) - 2 * (s.count("AB") + s.count("CD"))

print("sai :", sai_dem_cap("ACDB"))        # 4 - 2 = 2
print("đúng:", Solution().minLength("ACDB"))   # 0
assert sai_dem_cap("ACDB") != Solution().minLength("ACDB")

sai : 2
đúng: 0


**Vì sao sai:** `count` chỉ đếm cặp **đang có sẵn** trong chuỗi **ban đầu**, bỏ qua cặp **sinh ra sau khi xoá**. Với `"ACDB"`: có 1 `CD`, 0 `AB` nên ra `4 - 2 = 2`, nhưng sau khi xoá `CD` còn `AB` xoá tiếp → `0`.

### Lỗi 3: Quên kiểm tra stack rỗng trước khi xem đỉnh

In [25]:
def sai_quen_rong(s: str) -> int:
    stack = []
    for ch in s:
        if (stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D"):   # SAI: stack có thể rỗng
            stack.pop()
        else:
            stack.append(ch)
    return len(stack)

try:
    print(sai_quen_rong("ABFCACDB"))
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().minLength("ABFCACDB"))

IndexError: list index out of range
đúng: 2


**Vì sao sai:** ở ký tự đầu tiên stack còn rỗng, `stack[-1]` văng `IndexError`. Luôn viết `if stack and ...` (chú ý thứ tự: `stack` phải đứng **trước** thì `and` mới đoản mạch kịp).

### Lỗi 4: Đảo thứ tự cặp (so `"BA"` thay vì `"AB"`)

In [26]:
def sai_dao_cap(s: str) -> int:
    stack = []
    for ch in s:
        if stack and ((stack[-1] == "B" and ch == "A") or (stack[-1] == "D" and ch == "C")):   # SAI: ngược
            stack.pop()
        else:
            stack.append(ch)
    return len(stack)

print("sai :", sai_dao_cap("AB"), sai_dao_cap("BA"))                    # 2 0
print("đúng:", Solution().minLength("AB"), Solution().minLength("BA"))    # 0 2
assert sai_dao_cap("AB") != Solution().minLength("AB")

sai : 2 0
đúng: 0 2


**Vì sao sai:** cặp phải **A đứng trước B**, **C đứng trước D**. Đỉnh stack là ký tự đã đọc **trước** `ch`, nên đỉnh phải là `A` (hoặc `C`) còn `ch` là `B` (hoặc `D`). Hãy đọc to điều kiện thành lời trước khi code: "đỉnh là A và ký tự mới là B".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Cụm từ khoá: "**xoá chuỗi con**", "sau khi xoá các phần **nối lại**", "xoá cặp kề nhau" ⇒ **Stack**. Cùng họ: ngoặc hợp lệ (20), xoá ký tự trùng liền kề (1047), xoá chuỗi con trong chuỗi (1910), v.v.
- Quy trình stack chuẩn: **duyệt trái → phải; nếu ký tự hiện tại "khớp" đỉnh thì `pop`, ngược lại `push`.**
- Hỏi mình: "ký tự tiếp theo có thể *chấm dứt* một cặp với ký tự gần nhất chưa xoá không?" — stack là cấu trúc lưu "ký tự gần nhất chưa xoá".
- Đừng quên kiểm tra **stack rỗng** trước khi xem đỉnh.
- Nếu đề cho phép "xoá theo thứ tự bất kỳ" mà **đáp án vẫn duy nhất** thì nên nghĩ ngay đến greedy một lượt/stack thay vì thử mọi thứ tự.
- `replace` lặp lại là cách **đúng, dễ viết** khi `n` nhỏ — dùng làm oracle để kiểm tra cách nhanh.

## 15. Biến thể của bài

1. **Trả về chuỗi cuối cùng** thay vì độ dài: `"".join(stack)`. Ví dụ 1 trả `"FC"`.
2. **Nhiều cặp xoá hơn** (`"XY"`, `"PQ"`...): cách 4 với `dict`.
3. **LeetCode 1047** Remove All Adjacent Duplicates In String: xoá hai ký tự **giống nhau** liền kề; stack, so `stack[-1] == ch`.
4. **LeetCode 20** Valid Parentheses: ghép ngoặc đóng với ngoặc mở gần nhất.
5. **Đếm số cặp đã xoá**: `(n - len(stack)) // 2`.
6. **Xoá có chi phí** (mỗi lần xoá `AB` được điểm `x`, `CD` được điểm `y`): LeetCode 1717 — khi đó thứ tự xoá **có** ảnh hưởng (hai luật có điểm khác nhau), và phải dùng greedy hai lượt.

In [27]:
def final_string(s: str) -> str:
    stack = []
    for ch in s:
        if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

print(final_string("ABFCACDB"))        # FC
print(final_string("ACBBD"))           # ACBBD
n = len("ABFCACDB")
print("số cặp đã xoá:", (n - len(final_string("ABFCACDB"))) // 2)    # 3

FC
ACBBD
số cặp đã xoá: 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. lặp `replace` | xoá đến khi không còn cặp | `O(n²)` | `O(n)` | oracle, `n` nhỏ, hiểu đề |
| 2. stack | một lượt, đỉnh stack ghép cặp | `O(n)` | `O(n)` | **cách nộp bài khuyến nghị** |
| 3. con trỏ ghi tại chỗ | stack mượn mảng gốc | `O(n)` | `O(1)` phụ | ngôn ngữ có mảng sửa được, bộ nhớ chặt |
| 4. stack + `dict` | bảng cặp ký tự đóng → mở | `O(n)` | `O(n)` | nhiều cặp, dễ mở rộng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"AABB".replace("AB", "")`, `"ABAB".replace("AB", "")`, `[].pop()` có lỗi không, và `"AB" in ("AB", "CD")`.

**Bài 2 (tính tay).** Với `s = "CABDEAB"`, vẽ bảng stack từng bước và cho biết độ dài cuối.

**Bài 3 (điền TODO).** Hoàn thành `min_length_todo(s)` bằng stack.

In [28]:
def min_length_todo(s):
    stack = []
    for ch in s:
        # TODO 1: nếu stack không rỗng và (stack[-1], ch) là ("A", "B") hoặc ("C", "D")
        # TODO 2:     thì pop
        # TODO 3: ngược lại push ch
        pass
    # TODO 4: return số phần tử còn lại
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `min_length_inplace2(s)` dùng con trỏ `top` trên `list(s)` mà không nhìn Cách 3. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `remove_adjacent_duplicates(s)` (LeetCode 1047): xoá liên tục hai ký tự **giống nhau** kề nhau, trả chuỗi cuối. (b) Viết `is_fully_removable(s)` trả `True` nếu có thể xoá hết `s` bằng luật `AB`/`CD` (đáp án = `min_length == 0`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"AABB".replace("AB", "")` là `"AB"`; `"ABAB"` → `""`; `[].pop()` có lỗi `IndexError`; `"AB" in ("AB", "CD")` là `True` (kiểm tra thuộc tuple). Bài 2: xử lý lần lượt `C, A, B, D, E, A, B`.
- **Mức 2 (vừa):** Bài 3: `if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")): stack.pop()` rồi `else: stack.append(ch)`; cuối `return len(stack)`. Bài 4: `top` là kích thước stack; `arr[top - 1] + ch in ("AB", "CD")` để ghép cặp.
- **Mức 3 (sâu):** Bài 5(a): giống Cách 2 nhưng điều kiện là `stack[-1] == ch`. Bài 5(b): chỉ cần `Solution().minLength(s) == 0`; nhắc: độ dài `s` phải **chẵn**.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert "AABB".replace("AB", "") == "AB"
assert "ABAB".replace("AB", "") == ""
try:
    [].pop()
    raised = False
except IndexError:
    raised = True
assert raised
assert ("AB" in ("AB", "CD")) is True

# Bài 2: CABDEAB
# C -> [C]; A -> [C,A]; B -> ghép AB -> [C]; D -> ghép CD -> []; E -> [E]; A -> [E,A]; B -> ghép AB -> [E]
assert Solution().minLength("CABDEAB") == 1

# Bài 3
def min_length_todo_ans(s):
    stack = []
    for ch in s:
        if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
            stack.pop()
        else:
            stack.append(ch)
    return len(stack)

assert min_length_todo_ans("ABFCACDB") == 2
assert min_length_todo_ans("ACBBD") == 5

# Bài 4
def min_length_inplace2(s):
    arr = list(s)
    top = 0
    for i in range(len(arr)):
        if top > 0 and arr[top - 1] + arr[i] in ("AB", "CD"):
            top -= 1
        else:
            arr[top] = arr[i]
            top += 1
    return top

assert min_length_inplace2("ABFCACDB") == 2
assert min_length_inplace2("ACBBD") == 5

# Bài 5
def remove_adjacent_duplicates(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

assert remove_adjacent_duplicates("abbaca") == "ca"
assert remove_adjacent_duplicates("azxxzy") == "ay"

def is_fully_removable(s):
    return Solution().minLength(s) == 0

assert is_fully_removable("AABB") is True
assert is_fully_removable("ACBBD") is False
assert is_fully_removable("ABCD") is True
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (stack: ký tự khớp đỉnh thì pop, ngược lại push)

```python
class Solution:
    def minLength(self, s: str) -> int:
        stack = []
        for ch in s:
            if stack and ((stack[-1] == "A" and ch == "B") or (stack[-1] == "C" and ch == "D")):
                stack.pop()
            else:
                stack.append(ch)
        return len(stack)
```

**Checklist**

- [ ] Nhớ rằng sau khi xoá, hai phần **dính lại** và có thể tạo cặp mới (không dùng `replace` một lượt / `count`)?
- [ ] Kiểm tra `stack` **không rỗng** trước khi xem `stack[-1]`?
- [ ] Cặp đúng thứ tự: đỉnh `A` + ký tự mới `B`; đỉnh `C` + ký tự mới `D`?
- [ ] Khi ghép được thì **không** push ký tự mới?
- [ ] Trả về `len(stack)` (độ dài, không phải chuỗi)?
- [ ] Đã thử `"AB"`, `"BA"`, `"AABB"`, `"ACDB"`, `"ACBBD"` và chuỗi dài 100?
- [ ] Signature đúng `minLength(self, s: str) -> int`?